# Kannada OCR Post-Processing System — Evaluation Notebook
**Steps:** Load dataset → Predict corrections → Calculate CER/WER → Error-type analysis → Context metrics

## Cell 1 — Install Dependencies

In [1]:
import os
os.environ['DISABLE_GEMINI_TESTING'] = '1'
print('✅ Gemini disabled')

✅ Gemini disabled


In [2]:
# Install required libraries
import subprocess
subprocess.run(['pip', 'install', 'jiwer', 'pandas', 'openpyxl', '--quiet'])
print('✅ Dependencies ready')

✅ Dependencies ready


## Cell 2 — Imports and Configuration

In [3]:
import pandas as pd
import numpy as np
import sys
import os
import re
import Levenshtein as lev
from jiwer import cer, wer
from datetime import datetime

# ============================================================
# CONFIGURATION — Change paths here if needed
# ============================================================
INPUT_CSV_PATH   = r'F:\INTERFACE_GRADIO\data\metrics_cal_data.csv'
VALIDATOR_PATH   = r'F:\INTERFACE_GRADIO'
OUTPUT_EXCEL     = r'F:\INTERFACE_GRADIO\data\evaluation_results.xlsx'
OUTPUT_CSV       = r'F:\INTERFACE_GRADIO\data\evaluation_results.csv'

# Add validator to path
sys.path.insert(0, VALIDATOR_PATH)

print('✅ Imports done')
print(f'📂 Input: {INPUT_CSV_PATH}')
print(f'📂 Output: {OUTPUT_EXCEL}')

✅ Imports done
📂 Input: F:\INTERFACE_GRADIO\data\metrics_cal_data.csv
📂 Output: F:\INTERFACE_GRADIO\data\evaluation_results.xlsx


## Cell 3 — Load Validator and Dataset

In [4]:
import os
os.environ['DISABLE_GEMINI_TESTING'] = '1'
print('Gemini disabled for evaluation')

Gemini disabled for evaluation


In [5]:
# Disable entity memory building during batch evaluation
# It is designed for interactive UI use, not batch processing
from new_validator import ContextAnalyzer
ContextAnalyzer.build_entity_memory_from_paragraph = classmethod(lambda cls, text: None)
print('Entity memory disabled for batch evaluation')

f:\INTERFACE_GRADIO\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


[Gemini] ✅ Context awareness enabled
Entity memory disabled for batch evaluation


f:\INTERFACE_GRADIO\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
F:\INTERFACE_GRADIO\new_validator.py:24: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [6]:
import os
import sys
import pandas as pd

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

# Paths
INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\metrics_cal_data.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'

# Force add to path
if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

print(f'Python path includes: {VALIDATOR_PATH}')

# Load validator
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    VALIDATOR_AVAILABLE = True
    print('✅ EnhancedValidator loaded successfully')
except Exception as e:
    VALIDATOR_AVAILABLE = False
    validator = None
    print(f'❌ Validator failed: {e}')
    import traceback
    traceback.print_exc()

# Load dataset
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nValidator available: {VALIDATOR_AVAILABLE}')
df.head(3)

Python path includes: F:\INTERFACE_GRADIO
[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,946 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,946 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
✅ EnhancedValidator loaded successfully
✅ Dataset loaded: 3000 rows
Columns: ['ID', 'Incorrect Sentence', 'Correct Sentence', 'Error Type']

Validator available: True


,ID,Incorrect Sentence,Correct Sentence,Error Type
0,1,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದನು,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದಳು,Gender Agreement
1,2,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾನೆ,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾಳೆ,Gender Agreement
2,3,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದನು,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದಳು,Gender Agreement


In [43]:
# Load EnhancedValidator
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    VALIDATOR_AVAILABLE = True
    print('✅ EnhancedValidator loaded')
except Exception as e:
    VALIDATOR_AVAILABLE = False
    print(f'⚠️  Validator not available: {e}')
    print('   Will use rule-based prediction fallback')

# Load dataset
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'\n✅ Dataset loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
df.head()

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,946 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,946 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
✅ EnhancedValidator loaded

✅ Dataset loaded: 3000 rows
Columns: ['ID', 'Incorrect Sentence', 'Correct Sentence', 'Error Type']


,ID,Incorrect Sentence,Correct Sentence,Error Type
0,1,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದನು,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದಳು,Gender Agreement
1,2,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾನೆ,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾಳೆ,Gender Agreement
2,3,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದನು,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದಳು,Gender Agreement
3,4,ಪ್ರಸಿದ್ಧ ಗಾಯಕಿ ವೇದಿಕೆಯಲ್ಲಿ ಹಾಡಿದನು,ಪ್ರಸಿದ್ಧ ಗಾಯಕಿ ವೇದಿಕೆಯಲ್ಲಿ ಹಾಡಿದಳು,Gender Agreement
4,5,ಹೊಸ ಶಿಕ್ಷಕಿ ಮಕ್ಕಳಿಗೆ ಪಾಠ ಕಲಿಸಿದನು,ಹೊಸ ಶಿಕ್ಷಕಿ ಮಕ್ಕಳಿಗೆ ಪಾಠ ಕಲಿಸಿದಳು,Gender Agreement


## Cell 4 — Prediction Engine

In [7]:
# Speed fix — limit Levenshtein search for batch mode
import os
os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

# Get validator reference from Cell 3
try:
    _validator_ref = validator
    _validator_available = True
except NameError:
    _validator_ref = None
    _validator_available = False

def correct_sentence(incorrect_sentence: str) -> str:
    """
    Run the EnhancedValidator correction pipeline on a sentence.
    Returns the predicted corrected sentence.
    """
    if not incorrect_sentence or not isinstance(incorrect_sentence, str):
        return incorrect_sentence

    if not _validator_available or _validator_ref is None:
        return incorrect_sentence

    words  = incorrect_sentence.split()
    result = []

    for idx, word in enumerate(words):
        clean = word.strip('.,!?;:()[]{}"\'।')

        # Skip punctuation tokens and non-Kannada
        if not clean or not any('\u0C80' <= c <= '\u0CFF' for c in clean):
            result.append(word)
            continue

        try:
            prev_word = result[-1].strip('.,!?;:()[]{}"\'।') if result else None
            val_result = _validator_ref.validate_word(
                clean,
                prev_word=prev_word,
                full_sentence=incorrect_sentence,
                position=idx,
            )
            suggestions = val_result.get('suggestions', [])

            if suggestions and suggestions[0] != clean:
                corrected_word = suggestions[0]
                prefix = word[:len(word) - len(word.lstrip('.,!?;:()[]{}"\'।'))]
                suffix = word[len(word.rstrip('.,!?;:()[]{}"\'।')):]
                result.append(prefix + corrected_word + suffix)
            else:
                result.append(word)
        except Exception as e:
            result.append(word)

    return ' '.join(result)

print('✅ Prediction engine ready')
print(f'   Validator available: {_validator_available}')
print('\n--- Quick Test ---')
test = 'ಅವನು ಪುತ್ತಕ ಓದುತ್ತಾನೆ'
print(f'Input:     {test}')
print(f'Predicted: {correct_sentence(test)}')

✅ Prediction engine ready
   Validator available: True

--- Quick Test ---
Input:     ಅವನು ಪುತ್ತಕ ಓದುತ್ತಾನೆ
Predicted: ಅವನು ಪುಸ್ತಕ ಓದುತ್ತಾನೆ


## Cell 5 — Metric Calculation Functions

In [8]:
def calc_cer(reference: str, hypothesis: str) -> float:
    """Character Error Rate as percentage."""
    if not reference or not hypothesis:
        return 0.0
    try:
        return round(cer(reference, hypothesis) * 100, 2)
    except:
        dist = lev.distance(reference, hypothesis)
        return round((dist / len(reference)) * 100, 2)

def calc_wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate as percentage."""
    if not reference or not hypothesis:
        return 0.0
    try:
        return round(wer(reference, hypothesis) * 100, 2)
    except:
        ref_words = reference.split()
        dist = lev.distance(' '.join(ref_words), ' '.join(hypothesis.split()))
        return round((dist / max(len(ref_words), 1)) * 100, 2)

def is_exact_match(reference: str, hypothesis: str) -> int:
    """1 if exact match, 0 otherwise."""
    return 1 if reference.strip() == hypothesis.strip() else 0

# Context-aware metrics
MALE_VERB_SUFFIXES   = ['ದನು', 'ತ್ತಾನೆ', 'ಹೋದನು', 'ಬಂದನು', 'ದಾನೆ']
FEMALE_VERB_SUFFIXES = ['ದಳು', 'ತ್ತಾಳೆ', 'ಹೋದಳು', 'ಬಂದಳು', 'ದಾಳೆ']
PLURAL_VERB_SUFFIXES = ['ದರು', 'ತ್ತಾರೆ', 'ಹೋದರು', 'ಬಂದರು', 'ದಾರೆ']
HONORIFIC_SUBJECTS   = {'ತಾತ', 'ಅಜ್ಜಿ', 'ಡಾಕ್ಟರ್', 'ಗುರು', 'ಆಚಾರ್ಯ', 'ಶಿಕ್ಷಕ'}
MALE_PRONOUNS        = {'ಅವನು', 'ಈತನು', 'ಇವನು'}
FEMALE_PRONOUNS      = {'ಅವಳು', 'ಈಕೆ', 'ಇವಳು'}

def check_gender_agreement(sentence: str) -> bool:
    """Check if pronoun and following verb gender match."""
    words = sentence.split()
    for i, w in enumerate(words):
        clean = w.strip('.,!?;:()[]{}"\'।')
        if clean in MALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return False  # Mismatch
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return True
        elif clean in FEMALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return False  # Mismatch
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return True
    return True  # No mismatch found

def check_honorific_agreement(sentence: str) -> bool:
    """Check if honorific subject has respectful verb form."""
    words = [w.strip('.,!?;:()[]{}"\'।') for w in sentence.split()]
    has_honorific_subject = any(w in HONORIFIC_SUBJECTS for w in words)
    if not has_honorific_subject:
        return True  # Not applicable
    # Check if any verb uses non-honorific male/female singular form
    for w in words:
        if any(w.endswith(s) for s in ['ದನು', 'ದಳು', 'ತ್ತಾನೆ', 'ತ್ತಾಳೆ']):
            return False  # Should use plural/respectful form
    return True

def check_sov_order(sentence: str) -> bool:
    """Check if verb appears at end of sentence (SOV)."""
    words = [w.strip('.,!?;:()[]{}"\'।') for w in sentence.split()]
    words = [w for w in words if w]  # Remove empty
    if len(words) < 2:
        return True
    last_word = words[-1]
    all_verb_suffixes = MALE_VERB_SUFFIXES + FEMALE_VERB_SUFFIXES + PLURAL_VERB_SUFFIXES
    return any(last_word.endswith(s) for s in all_verb_suffixes)

print('✅ Metric functions ready')

✅ Metric functions ready


## Cell 6 — Run Predictions on Full Dataset

In [9]:
import os
import numpy as np
import pandas as pd

os.environ['DISABLE_GEMINI_TESTING'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\metrics_cal_data.csv'

# Reload df if not defined
try:
    df
    print(f'✅ df already loaded: {len(df)} rows')
except NameError:
    df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
    print(f'✅ df reloaded: {len(df)} rows')

# Reload validator if not defined
try:
    validator
    _validator_ok = True
    print('✅ Validator already loaded')
except NameError:
    _validator_ok = False
    print('⚠️  Validator not loaded — run Cell 3 first')

print(f'\nProcessing {len(df)} rows...')
print('Valid words = instant. Invalid words = ~10s each.\n')

predicted_sentences = []
is_correct_list     = []
cer_list            = []
wer_list            = []

for i, row in df.iterrows():
    incorrect = str(row['Incorrect Sentence']).strip()
    correct   = str(row['Correct Sentence']).strip()

    # Skip entity memory during batch — causes noise
    # ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)

    # Get prediction
    predicted = correct_sentence(incorrect) if _validator_ok else incorrect

    # Calculate metrics
    is_corr = is_exact_match(correct, predicted)
    c_rate  = calc_cer(correct, predicted)
    w_rate  = calc_wer(correct, predicted)

    predicted_sentences.append(predicted)
    is_correct_list.append(is_corr)
    cer_list.append(c_rate)
    wer_list.append(w_rate)

    # Progress every 50 rows
    if (i + 1) % 50 == 0 or (i + 1) == len(df):
        acc_so_far = sum(is_correct_list) / len(is_correct_list) * 100
        print(f'  [{i+1}/{len(df)}] done — running accuracy: {acc_so_far:.1f}%')

# Add columns to dataframe
df['Predicted Sentence'] = predicted_sentences
df['Is Correct']         = is_correct_list
df['CER']                = cer_list
df['WER']                = wer_list

print(f'\n✅ Predictions complete')
print(f'Correct: {sum(is_correct_list)}/{len(df)} ({sum(is_correct_list)/len(df)*100:.1f}%)')
print(f'Avg CER: {np.mean(cer_list):.2f}%')
print(f'Avg WER: {np.mean(wer_list):.2f}%')

✅ df already loaded: 3000 rows
✅ Validator already loaded

Processing 3000 rows...
Valid words = instant. Invalid words = ~10s each.



KeyboardInterrupt: 

In [10]:
import os
import sys
import numpy as np
import pandas as pd
import Levenshtein as lev

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\metrics_cal_data.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'

# Add path
if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset loaded: {len(df)} rows')
print(f'   Columns: {list(df.columns)}')

# ── Load validator ────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    VALIDATOR_OK = True
    print('✅ Validator loaded')
except Exception as e:
    VALIDATOR_OK = False
    validator = None
    print(f'❌ Validator failed: {e}')

# ── Metric functions ──────────────────────────────────────
try:
    from jiwer import cer as jiwer_cer, wer as jiwer_wer
    JIWER_OK = True
except:
    JIWER_OK = False

def calc_cer(reference, hypothesis):
    if not reference or not hypothesis:
        return 0.0
    try:
        if JIWER_OK:
            return round(jiwer_cer(reference, hypothesis) * 100, 2)
        return round(lev.distance(reference, hypothesis) / len(reference) * 100, 2)
    except:
        return 0.0

def calc_wer(reference, hypothesis):
    if not reference or not hypothesis:
        return 0.0
    try:
        if JIWER_OK:
            return round(jiwer_wer(reference, hypothesis) * 100, 2)
        ref_words = reference.split()
        dist = lev.distance(' '.join(ref_words), ' '.join(hypothesis.split()))
        return round(dist / max(len(ref_words), 1) * 100, 2)
    except:
        return 0.0

def is_exact_match(reference, hypothesis):
    return 1 if str(reference).strip() == str(hypothesis).strip() else 0

# ── Correction function ───────────────────────────────────
def correct_sentence(sentence):
    if not sentence or not isinstance(sentence, str):
        return sentence
    if not VALIDATOR_OK or validator is None:
        return sentence

    words  = sentence.split()
    result = []

    for idx, word in enumerate(words):
        clean = word.strip('.,!?;:()[]{}"\'।')
        if not clean or not any('\u0C80' <= c <= '\u0CFF' for c in clean):
            result.append(word)
            continue
        try:
            prev_word  = result[-1].strip('.,!?;:()[]{}"\'।') if result else None
            val_result = validator.validate_word(
                clean,
                prev_word=prev_word,
                full_sentence=sentence,
                position=idx,
            )
            suggestions = val_result.get('suggestions', [])
            if suggestions and suggestions[0] != clean:
                prefix = word[:len(word) - len(word.lstrip('.,!?;:()[]{}"\'।'))]
                suffix = word[len(word.rstrip('.,!?;:()[]{}"\'।')):]
                result.append(prefix + suggestions[0] + suffix)
            else:
                result.append(word)
        except:
            result.append(word)

    return ' '.join(result)

# ── Quick test ────────────────────────────────────────────
test = 'ಅವನು ಪುತ್ತಕ ಓದುತ್ತಾನೆ'
print(f'\nQuick test:')
print(f'  Input:     {test}')
print(f'  Predicted: {correct_sentence(test)}')
print(f'  Validator: {VALIDATOR_OK}')

# ── Run predictions ───────────────────────────────────────
print(f'\nProcessing {len(df)} rows...\n')

predicted_sentences = []
is_correct_list     = []
cer_list            = []
wer_list            = []

for i, row in df.iterrows():
    incorrect = str(row['Incorrect Sentence']).strip()
    correct   = str(row['Correct Sentence']).strip()
    predicted = correct_sentence(incorrect)

    predicted_sentences.append(predicted)
    is_correct_list.append(is_exact_match(correct, predicted))
    cer_list.append(calc_cer(correct, predicted))
    wer_list.append(calc_wer(correct, predicted))

    if (i + 1) % 50 == 0 or (i + 1) == len(df):
        acc = sum(is_correct_list) / len(is_correct_list) * 100
        print(f'  [{i+1}/{len(df)}] — accuracy so far: {acc:.1f}%')

# Add to dataframe
df['Predicted Sentence'] = predicted_sentences
df['Is Correct']         = is_correct_list
df['CER']                = cer_list
df['WER']                = wer_list

print(f'\n✅ Done')
print(f'Correct:  {sum(is_correct_list)}/{len(df)} ({sum(is_correct_list)/len(df)*100:.1f}%)')
print(f'Avg CER:  {np.mean(cer_list):.2f}%')
print(f'Avg WER:  {np.mean(wer_list):.2f}%')

✅ Dataset loaded: 3000 rows
   Columns: ['ID', 'Incorrect Sentence', 'Correct Sentence', 'Error Type']
[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,946 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,946 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
✅ Validator loaded

Quick test:
  Input:     ಅವನು ಪುತ್ತಕ ಓದುತ್ತಾನೆ
  Predicted: ಅವನು ಪುಸ್ತಕ ಓದುತ್ತಾನೆ
  Validator: True

Processing 3000 rows...

  [50/3000] — accuracy so far: 2.0%
  [100/3000] — accuracy so far: 2.0%
  [150/3000] — accuracy so far: 1.3%
  [200/3000] — accuracy so far: 3.0%
  [250/3000] — accuracy so far: 2.4%
  [300/3000] — accuracy so far: 3.3%
  [350/3000] — accuracy so far: 2.9%
  [400/3000] — accuracy so far: 2.5%
  [450/3000] — accuracy so far: 2.4%
  [500/3000] — accuracy so far: 2.8%
  [550/3000] — accuracy so far: 2.9%
  [600/3000] — accuracy so far: 3.2%
  [650/3000]

## Cell 7 — Overall Summary Metrics

In [11]:
total        = len(df)
total_correct= sum(df['Is Correct'])
word_acc     = round(total_correct / total * 100, 2)
avg_cer      = round(df['CER'].mean(), 2)
avg_wer      = round(df['WER'].mean(), 2)
char_acc     = round(100 - avg_cer, 2)
word_acc_from_wer = round(100 - avg_wer, 2)

summary_data = [
    ['Total Samples',              total],
    ['Total Correct Predictions',  total_correct],
    ['Sentence Accuracy (%)',      word_acc],
    ['Average CER (%)',            avg_cer],
    ['Average WER (%)',            avg_wer],
    ['Character Accuracy (%)',     char_acc],
    ['Word Accuracy from WER (%)', word_acc_from_wer],
    ['Min CER (%)',                round(df['CER'].min(), 2)],
    ['Max CER (%)',                round(df['CER'].max(), 2)],
    ['Min WER (%)',                round(df['WER'].min(), 2)],
    ['Max WER (%)',                round(df['WER'].max(), 2)],
]

summary_df = pd.DataFrame(summary_data, columns=['Metric', 'Value'])
print('='*45)
print('OVERALL SUMMARY METRICS')
print('='*45)
print(summary_df.to_string(index=False))
print('='*45)

OVERALL SUMMARY METRICS
                    Metric   Value
             Total Samples 3000.00
 Total Correct Predictions  102.00
     Sentence Accuracy (%)    3.40
           Average CER (%)   18.36
           Average WER (%)   37.57
    Character Accuracy (%)   81.64
Word Accuracy from WER (%)   62.43
               Min CER (%)    0.00
               Max CER (%)  273.91
               Min WER (%)    0.00
               Max WER (%)  275.00


## Cell 8 — Error-Type-Wise Metrics

In [12]:
error_type_col = 'Error Type'  # Change if your column name is different

if error_type_col not in df.columns:
    print(f'⚠️  Column "{error_type_col}" not found.')
    print(f'Available columns: {list(df.columns)}')
else:
    error_groups = df.groupby(error_type_col)
    error_rows   = []

    for error_type, group in error_groups:
        count    = len(group)
        accuracy = round(group['Is Correct'].sum() / count * 100, 2)
        avg_c    = round(group['CER'].mean(), 2)
        avg_w    = round(group['WER'].mean(), 2)
        error_rows.append([error_type, count, f'{accuracy}%', f'{avg_c}%', f'{avg_w}%'])

    error_df = pd.DataFrame(error_rows,
                            columns=['Error Type', 'Count', 'Accuracy', 'Avg CER', 'Avg WER'])
    print('='*65)
    print('ERROR-TYPE-WISE METRICS')
    print('='*65)
    print(error_df.to_string(index=False))
    print('='*65)

ERROR-TYPE-WISE METRICS
                Error Type  Count Accuracy Avg CER Avg WER
         Case Marker Error    300     0.0%  17.62%  31.91%
              Extra Suffix    300    32.0%  10.82%  26.55%
          Gender Agreement    300     0.0%   8.31%  37.89%
       Honorific Agreement    300    0.33%   7.58%  34.73%
          Neuter Agreement    300    0.33%   15.1%   38.4%
          Number Agreement    300     1.0%   7.08%  35.91%
          Person Agreement    300    0.33%  12.14%  33.61%
        Postposition Error    300     0.0%  18.61%  36.16%
           SOV Order Error    300     0.0%  59.01%  63.86%
               Tense Error    299     0.0%  27.16%   36.5%
ಮುಂದಿನ ತಿಂಗಳು ಮಳೆ ಬರುತ್ತದೆ      1     0.0%  80.58%  88.24%


## Cell 9 — Context Metrics (Agreement + SOV + Honorific)

In [13]:
gender_correct_pred   = []
gender_correct_true   = []
honorific_correct_pred= []
honorific_correct_true= []
sov_correct_pred      = []
sov_correct_true      = []

for _, row in df.iterrows():
    pred    = str(row['Predicted Sentence'])
    correct = str(row['Correct Sentence'])

    gender_correct_pred.append(1 if check_gender_agreement(pred)    else 0)
    gender_correct_true.append(1 if check_gender_agreement(correct)  else 0)
    honorific_correct_pred.append(1 if check_honorific_agreement(pred)   else 0)
    honorific_correct_true.append(1 if check_honorific_agreement(correct) else 0)
    sov_correct_pred.append(1 if check_sov_order(pred)    else 0)
    sov_correct_true.append(1 if check_sov_order(correct)  else 0)

def safe_acc(pred_list, true_list):
    """Accuracy: how often prediction matches ground truth on this metric."""
    matches = sum(p == t for p, t in zip(pred_list, true_list))
    return round(matches / len(pred_list) * 100, 2) if pred_list else 0.0

context_data = [
    ['Gender Agreement Accuracy',    f'{safe_acc(gender_correct_pred, gender_correct_true)}%'],
    ['Honorific Agreement Accuracy', f'{safe_acc(honorific_correct_pred, honorific_correct_true)}%'],
    ['SOV Order Accuracy',           f'{safe_acc(sov_correct_pred, sov_correct_true)}%'],
    ['Gender OK in Predictions',     f'{round(sum(gender_correct_pred)/len(gender_correct_pred)*100,2)}%'],
    ['Honorific OK in Predictions',  f'{round(sum(honorific_correct_pred)/len(honorific_correct_pred)*100,2)}%'],
    ['SOV OK in Predictions',        f'{round(sum(sov_correct_pred)/len(sov_correct_pred)*100,2)}%'],
]

context_df = pd.DataFrame(context_data, columns=['Context Metric', 'Value'])
print('='*50)
print('CONTEXT METRICS')
print('='*50)
print(context_df.to_string(index=False))
print('='*50)

CONTEXT METRICS
              Context Metric  Value
   Gender Agreement Accuracy 100.0%
Honorific Agreement Accuracy  99.2%
          SOV Order Accuracy  73.3%
    Gender OK in Predictions 100.0%
 Honorific OK in Predictions 99.17%
       SOV OK in Predictions  55.8%


## Cell 10 — Full Output Table Preview

In [14]:
# Reorder columns for clean display
display_cols = ['ID', 'Incorrect Sentence', 'Correct Sentence',
                'Predicted Sentence', 'Error Type', 'Is Correct', 'CER', 'WER']

# Only keep columns that exist
display_cols = [c for c in display_cols if c in df.columns]
full_table   = df[display_cols].copy()

print(f'Full table: {len(full_table)} rows × {len(display_cols)} columns')
full_table.head(10)

Full table: 3000 rows × 8 columns


,ID,Incorrect Sentence,Correct Sentence,Predicted Sentence,Error Type,Is Correct,CER,WER
0,1,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದನು,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದಳು,ರಮ್ಯಾ ಹೊಸ ಪುಸ್ತಕವನ್ನು ಓದಿದನು,Gender Agreement,0,3.57,25.0
1,2,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾನೆ,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾಳೆ,ಅಮ್ಮ ರುಚಿಯಾದ ಅಡುಗೆ ಮಾಡುತ್ತಿದ್ದಾನೆ,Gender Agreement,0,3.03,25.0
2,3,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದನು,ಸಣ್ಣ ಹುಡುಗಿ ವೇಗವಾಗಿ ಓಡಿದಳು,ಸಣ್ಣಕ್ಕೆ ಹುಡುಗಿಗೆ ವೇಗವಾಗಿಗೆ ಓಡಿದನು,Gender Agreement,0,34.62,100.0
3,4,ಪ್ರಸಿದ್ಧ ಗಾಯಕಿ ವೇದಿಕೆಯಲ್ಲಿ ಹಾಡಿದನು,ಪ್ರಸಿದ್ಧ ಗಾಯಕಿ ವೇದಿಕೆಯಲ್ಲಿ ಹಾಡಿದಳು,ಪ್ರಸಿದ್ಧ ಗಾಯಕಿ ವೇದಿಕೆಯಲ್ಲಿ ಹಾಡಿದನು,Gender Agreement,0,2.94,25.0
4,5,ಹೊಸ ಶಿಕ್ಷಕಿ ಮಕ್ಕಳಿಗೆ ಪಾಠ ಕಲಿಸಿದನು,ಹೊಸ ಶಿಕ್ಷಕಿ ಮಕ್ಕಳಿಗೆ ಪಾಠ ಕಲಿಸಿದಳು,ಹೊಸ ಶಿಕ್ಷಕಿ ಮಕ್ಕಳಿಗೆ ಪಾಠ ಕಲಿಸಿದನು,Gender Agreement,0,3.03,20.0
5,6,ಎಲ್ಲಾ ಮಕ್ಕಳು ಮೈದಾನದಲ್ಲಿ ಆಡುತ್ತಾನೆ,ಎಲ್ಲಾ ಮಕ್ಕಳು ಮೈದಾನದಲ್ಲಿ ಆಡುತ್ತಾರೆ,ಎಲ್ಲಾಅಲ್ಲಿ ಮಕ್ಕಳು ಮೈದಾನದಲ್ಲಿ ಆಡುತ್ತಾನೆ,Number Agreement,0,18.18,50.0
6,7,ವಿದ್ಯಾರ್ಥಿಗಳು ಪರೀಕ್ಷೆಯನ್ನು ಚೆನ್ನಾಗಿ ಬರೆದನು,ವಿದ್ಯಾರ್ಥಿಗಳು ಪರೀಕ್ಷೆಯನ್ನು ಚೆನ್ನಾಗಿ ಬರೆದರು,ವಿದ್ಯಾರ್ಥಿಗಳು ಪರೀಕ್ಷೆಯನ್ನು ಚೆನ್ನಾಗಿಗೆ ಬರೆದನು,Number Agreement,0,7.14,50.0
7,8,ಸುಂದರವಾದ ಹಕ್ಕಿಗಳು ಆಕಾಶದಲ್ಲಿ ಹಾರುತ್ತದೆ,ಸುಂದರವಾದ ಹಕ್ಕಿಗಳು ಆಕಾಶದಲ್ಲಿ ಹಾರುತ್ತವೆ,ಸುಂದರವಾದ ಹಕ್ಕಿಗಳು ಆಕಾಶದಲ್ಲಿ ಹಾರುತ್ತದೆ,Number Agreement,0,2.70,25.0
8,9,ಹಳ್ಳಿಯ ಜನರು ಜಾತ್ರೆಗೆ ಹೋದನು,ಹಳ್ಳಿಯ ಜನರು ಜಾತ್ರೆಗೆ ಹೋದರು,ಹಳ್ಳಿಯಕ್ಕೆ ಜನರಕ್ಕೆ ಜಾತ್ರೆಗೆ ಹೋದನು,Number Agreement,0,34.62,75.0
9,10,ಕಪ್ಪು ಹಸುಗಳು ಹೊಲದಲ್ಲಿ ಹುಲ್ಲು ಮೇಯುತ್ತದೆ,ಕಪ್ಪು ಹಸುಗಳು ಹೊಲದಲ್ಲಿ ಹುಲ್ಲು ಮೇಯುತ್ತವೆ,ಕಪ್ಪು ಹಸುಗಳು ಹೊಲದಲ್ಲಿ ಹುಲ್ಲು ಮೇಯುತ್ತದೆ,Number Agreement,0,2.63,20.0


## Cell 11 — Export to Excel and CSV

In [15]:
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    # Sheet 1: Full results table
    full_table.to_excel(writer, sheet_name='Full Results', index=False)

    # Sheet 2: Summary metrics
    summary_df.to_excel(writer, sheet_name='Summary Metrics', index=False)

    # Sheet 3: Error-type metrics
    if 'error_df' in dir():
        error_df.to_excel(writer, sheet_name='Error Type Metrics', index=False)

    # Sheet 4: Context metrics
    context_df.to_excel(writer, sheet_name='Context Metrics', index=False)

# Also save CSV
full_table.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ Excel saved: {OUTPUT_EXCEL}')
print(f'✅ CSV saved:   {OUTPUT_CSV}')
print(f'\nSheets in Excel:')
print('  1. Full Results')
print('  2. Summary Metrics')
print('  3. Error Type Metrics')
print('  4. Context Metrics')

✅ Excel saved: F:\INTERFACE_GRADIO\data\evaluation_results.xlsx
✅ CSV saved:   F:\INTERFACE_GRADIO\data\evaluation_results.csv

Sheets in Excel:
  1. Full Results
  2. Summary Metrics
  3. Error Type Metrics
  4. Context Metrics


## Cell 12 — Final Report Print (Copy-paste ready)

In [16]:
print('='*60)
print('KANNADA OCR POST-PROCESSING SYSTEM — EVALUATION REPORT')
print(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*60)

print('\n[1] OVERALL METRICS')
print(f'    Total Samples         : {total}')
print(f'    Correct Predictions   : {total_correct}')
print(f'    Sentence Accuracy     : {word_acc}%')
print(f'    Average CER           : {avg_cer}%')
print(f'    Average WER           : {avg_wer}%')
print(f'    Character Accuracy    : {char_acc}%')
print(f'    Word Accuracy (WER)   : {word_acc_from_wer}%')

print('\n[2] ERROR-TYPE BREAKDOWN')
if 'error_df' in dir():
    print(error_df.to_string(index=False))

print('\n[3] CONTEXT METRICS')
print(context_df.to_string(index=False))

print('\n' + '='*60)

KANNADA OCR POST-PROCESSING SYSTEM — EVALUATION REPORT
Generated: 2026-04-21 19:54:29

[1] OVERALL METRICS
    Total Samples         : 3000
    Correct Predictions   : 102
    Sentence Accuracy     : 3.4%
    Average CER           : 18.36%
    Average WER           : 37.57%
    Character Accuracy    : 81.64%
    Word Accuracy (WER)   : 62.43%

[2] ERROR-TYPE BREAKDOWN
                Error Type  Count Accuracy Avg CER Avg WER
         Case Marker Error    300     0.0%  17.62%  31.91%
              Extra Suffix    300    32.0%  10.82%  26.55%
          Gender Agreement    300     0.0%   8.31%  37.89%
       Honorific Agreement    300    0.33%   7.58%  34.73%
          Neuter Agreement    300    0.33%   15.1%   38.4%
          Number Agreement    300     1.0%   7.08%  35.91%
          Person Agreement    300    0.33%  12.14%  33.61%
        Postposition Error    300     0.0%  18.61%  36.16%
           SOV Order Error    300     0.0%  59.01%  63.86%
               Tense Error    299     0

In [18]:
import os
import sys
import numpy as np
import pandas as pd

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\metrics_cal_data.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Reload validator ──────────────────────────────────────
try:
    validator
    print('✅ Validator already loaded')
except NameError:
    from new_validator import EnhancedValidator
    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    print('✅ Validator loaded')

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')

# ── Find differing words between two sentences ────────────
def get_differing_words(incorrect, correct):
    """
    Returns list of (incorrect_word, correct_word) pairs
    where the two sentences differ at word level.
    """
    inc_words = incorrect.strip().split()
    cor_words = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc_words, cor_words)):
        iw_clean = iw.strip('.,!?;:()[]{}"\'।')
        cw_clean = cw.strip('.,!?;:()[]{}"\'।')
        if iw_clean != cw_clean and iw_clean and cw_clean:
            pairs.append((iw_clean, cw_clean, i, incorrect))
    return pairs

# ── Run suggestion evaluation ─────────────────────────────
print('\nRunning suggestion evaluation...\n')

top1 = top3 = top5 = top10 = 0
total_words   = 0
total_flagged = 0
not_flagged   = 0

error_type_results = {}

rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row.get('Error Type', 'Unknown')).strip()

    pairs = get_differing_words(incorrect, correct)

    for inc_word, cor_word, position, sentence in pairs:
        total_words += 1

        try:
            result      = validator.validate_word(
                inc_word,
                full_sentence=sentence,
                position=position
            )
            suggestions = result.get('suggestions', [])
            is_flagged  = not result.get('valid', True)
        except Exception as e:
            suggestions = []
            is_flagged  = False

        if is_flagged:
            total_flagged += 1
        else:
            not_flagged += 1

        in_top1  = 1 if cor_word in suggestions[:1]  else 0
        in_top3  = 1 if cor_word in suggestions[:3]  else 0
        in_top5  = 1 if cor_word in suggestions[:5]  else 0
        in_top10 = 1 if cor_word in suggestions[:10] else 0

        top1  += in_top1
        top3  += in_top3
        top5  += in_top5
        top10 += in_top10

        # Per error type tracking
        if error_type not in error_type_results:
            error_type_results[error_type] = {
                'total': 0, 'flagged': 0,
                'top1': 0, 'top3': 0, 'top5': 0, 'top10': 0
            }
        error_type_results[error_type]['total']   += 1
        error_type_results[error_type]['flagged'] += 1 if is_flagged else 0
        error_type_results[error_type]['top1']    += in_top1
        error_type_results[error_type]['top3']    += in_top3
        error_type_results[error_type]['top5']    += in_top5
        error_type_results[error_type]['top10']   += in_top10

        rows.append({
            'Incorrect Word':  inc_word,
            'Correct Word':    cor_word,
            'Error Type':      error_type,
            'Flagged':         is_flagged,
            'Suggestions':     str(suggestions[:5]),
            'In Top-1':        in_top1,
            'In Top-3':        in_top3,
            'In Top-5':        in_top5,
            'In Top-10':       in_top10,
        })

    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/{len(df)}] processed — '
              f'Top-1: {top1}/{total_words} '
              f'({top1/total_words*100:.1f}%)')

# ── Print Summary ─────────────────────────────────────────
print('\n' + '='*55)
print('SUGGESTION EVALUATION RESULTS')
print('='*55)
print(f'Total incorrect words evaluated : {total_words}')
print(f'Words flagged by system         : {total_flagged} ({total_flagged/total_words*100:.1f}%)')
print(f'Words NOT flagged (missed)      : {not_flagged} ({not_flagged/total_words*100:.1f}%)')
print(f'\nSuggestion Accuracy:')
print(f'  Top-1  (best suggestion correct) : {top1/total_words*100:.1f}%')
print(f'  Top-3  (correct in top 3)        : {top3/total_words*100:.1f}%')
print(f'  Top-5  (correct in top 5)        : {top5/total_words*100:.1f}%')
print(f'  Top-10 (correct in top 10)       : {top10/total_words*100:.1f}%')
print('='*55)

# ── Error type breakdown ──────────────────────────────────
print('\nERROR-TYPE-WISE SUGGESTION ACCURACY')
print('='*75)
print(f"{'Error Type':<30} {'Total':>6} {'Flagged':>8} {'Top-1':>7} {'Top-3':>7} {'Top-5':>7} {'Top-10':>8}")
print('-'*75)
for et, v in sorted(error_type_results.items()):
    t = v['total']
    print(f"{et:<30} {t:>6} "
          f"{v['flagged']/t*100:>7.1f}% "
          f"{v['top1']/t*100:>6.1f}% "
          f"{v['top3']/t*100:>6.1f}% "
          f"{v['top5']/t*100:>6.1f}% "
          f"{v['top10']/t*100:>7.1f}%")
print('='*75)

# ── Save results ──────────────────────────────────────────
results_df = pd.DataFrame(rows)
OUTPUT_PATH = r'F:\INTERFACE_GRADIO\data\suggestion_evaluation.xlsx'
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Word Level Results', index=False)

    summary = pd.DataFrame([
        ['Total Words Evaluated',       total_words],
        ['Words Flagged',               total_flagged],
        ['Words Missed',                not_flagged],
        ['Detection Rate (%)',          round(total_flagged/total_words*100, 2)],
        ['Top-1 Accuracy (%)',          round(top1/total_words*100, 2)],
        ['Top-3 Accuracy (%)',          round(top3/total_words*100, 2)],
        ['Top-5 Accuracy (%)',          round(top5/total_words*100, 2)],
        ['Top-10 Accuracy (%)',         round(top10/total_words*100, 2)],
    ], columns=['Metric', 'Value'])
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f'\n✅ Saved to {OUTPUT_PATH}')

✅ Validator already loaded
✅ Dataset: 3000 rows

Running suggestion evaluation...

  [100/3000] processed — Top-1: 8/130 (6.2%)
  [200/3000] processed — Top-1: 13/269 (4.8%)
  [300/3000] processed — Top-1: 18/408 (4.4%)
  [400/3000] processed — Top-1: 18/508 (3.5%)
  [500/3000] processed — Top-1: 25/669 (3.7%)
  [600/3000] processed — Top-1: 31/800 (3.9%)


KeyboardInterrupt: 

In [19]:
# Debug — test 5 sample rows to see what is happening
import os
os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

df_sample = df.head(20)

for i, row in df_sample.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    
    pairs = get_differing_words(incorrect, correct)
    
    if not pairs:
        print(f"Row {i}: NO DIFFERING WORDS FOUND")
        print(f"  Incorrect: {incorrect}")
        print(f"  Correct:   {correct}")
        continue
    
    for inc_word, cor_word, pos, sent in pairs:
        result      = validator.validate_word(inc_word, full_sentence=sent, position=pos)
        suggestions = result.get('suggestions', [])
        is_valid    = result.get('valid', True)
        
        print(f"Row {i} [{error_type}]")
        print(f"  Wrong word : {inc_word}")
        print(f"  Correct    : {cor_word}")
        print(f"  Valid?     : {is_valid}")
        print(f"  Top-5 sugg : {suggestions[:5]}")
        print(f"  Correct in suggestions: {cor_word in suggestions}")
        print()

Row 0 [Gender Agreement]
  Wrong word : ಓದಿದನು
  Correct    : ಓದಿದಳು
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 1 [Gender Agreement]
  Wrong word : ಮಾಡುತ್ತಿದ್ದಾನೆ
  Correct    : ಮಾಡುತ್ತಿದ್ದಾಳೆ
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 2 [Gender Agreement]
  Wrong word : ಓಡಿದನು
  Correct    : ಓಡಿದಳು
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 3 [Gender Agreement]
  Wrong word : ಹಾಡಿದನು
  Correct    : ಹಾಡಿದಳು
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 4 [Gender Agreement]
  Wrong word : ಕಲಿಸಿದನು
  Correct    : ಕಲಿಸಿದಳು
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 5 [Number Agreement]
  Wrong word : ಆಡುತ್ತಾನೆ
  Correct    : ಆಡುತ್ತಾರೆ
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions: False

Row 6 [Number Agreement]
  Wrong word : ಬರೆದನು
  Correct    : ಬರೆದರು
  Valid?     : True
  Top-5 sugg : []
  Correct in suggestions:

In [20]:
# Only evaluate error types your system is designed for
RELEVANT_ERROR_TYPES = [
    'Extra Suffix',
    'Case Marker Error', 
    'Postposition Error',
]

df_relevant = df[df['Error Type'].isin(RELEVANT_ERROR_TYPES)]
print(f'Relevant rows: {len(df_relevant)} out of {len(df)}')
print(df_relevant['Error Type'].value_counts())

Relevant rows: 900 out of 3000
Error Type
Case Marker Error     300
Postposition Error    300
Extra Suffix          300
Name: count, dtype: int64
